# Lab 11-1 - Feature Engineering

This lab covers the topic of "feature engineering" which is a broad term referring to the creation or extraction of predictive features from raw data to be used when training a downstream model.

In [ ]:
# Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

# turn off future warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  

# Use this seed wherever randomness is involved, for reproducibility of results
SEED = 76

This lab will use several different data sets for the purposes of providing a diverse range of examples. Each data set will be introduced in the part of the lab where it is used.

## Part 1 - One-hot Encoding

Up until now we've been avoiding the use of categorical predictors in our models; however, categorical predictors can be included into any model using a strategy known as "one-hot encoding" or "dummy variables" in the language of statistical modeling. The example provides a quick demonstration of one-hot encoding on the `'style'` and `'ac'` variables in the Iowa City home sales data:

![](../images/one_hot_encoding.png)

In [ ]:
# Load the Iowa City home sales data
ic = pd.read_csv(filepath_or_buffer = "../data/IowaCityHomeSales.csv")

# We'll one-hot encode these categorical vars:
encode_list = ['style', 'ac']

# Preview
ic[encode_list].head()

In [ ]:
# Frequency table of the categories of style, sorted alphanumerically
ic['style'].value_counts().sort_index()

In [ ]:
# Frequency table of the categories of ac, sorted alphanumerically
print(ic['ac'].value_counts())

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html 
from pandas import get_dummies

# Keep y = sale.amount, x1 = assessed value (numerical), x2 = style (categorical), x2 = ac (categorical)
ic_ohe = get_dummies(
    data = ic[['sale.amount','assessed','style','ac']], 
    columns = encode_list
)
print(ic_ohe.shape)
ic_ohe.head()

Notice how the numeric variables, `'sale.amount'` and `'assessed'`, were unchanged, but the variables `'style'` and `'ac'` were expanded using a collection of binary integer variables representing each unique value.

For some models, it's problematic to use linearly dependent sets of predictors, so we might opt to use "reference coding" via the `drop_first` argument to make the first category of each the encoded variables the reference category, thereby ensuring the new columns are not linearly dependent:

In [ ]:
# Keep outcome var sale.amount, numerical var assessed value, plus categorical variables to one-hot encode
# But also drop first column
ic_ohe = get_dummies(
    data = ic[['sale.amount','assessed','style','ac']], 
    columns=encode_list, 
    drop_first=True
)
print(ic_ohe.shape)
ic_ohe.head()

Note that the `style == 1 1/2 Story Frame` and `ac == No` are now encoded as reference categories in the sense that this category of home corresponds with all of the dummy variables being zero. This is evidenced by the number of columns dropping from 13 to 11.

Using one-hot encoding in a data processing pipeline is somewhat complicated, as it requires us to define different actions for numeric and categorical predictors:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

# Set up training data for this example
train_ic, test_ic = train_test_split(ic, test_size=0.2, random_state=7)
train_ic_y = train_ic['sale.amount']
train_ic_X = train_ic.drop(labels = 'sale.amount', axis = 1)

In [ ]:
# Seperate pipelines for numeric vs. cat vars
num_transformer = Pipeline(steps = [
    ("scaler", StandardScaler())
])
cat_transformer = Pipeline(steps = [
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

# Get names of numeric and categorical columns
num_col_names = train_ic_X.select_dtypes(exclude=['object']).columns.tolist()
cat_col_names = train_ic_X.select_dtypes(include=['object']).columns.tolist()

print("Numeric variables: " + str(num_col_names))
print("Categorical variables: " + str(cat_col_names))

In [ ]:
# Preprocessing transformer ColumnTransformer() allows different actions for numeric and categorical vars
preprocessor = ColumnTransformer(transformers = [
    ('num', num_transformer, num_col_names),
    ('cat', cat_transformer, cat_col_names)
])

# Combine the preprocessor and model into a final pipeline
final_pipe = Pipeline(steps = [
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier())
])

# We can fit this pipeline
fitted_pipe = final_pipe.fit(X = train_ic_X, y = train_ic_y)

The main purpose of pipelines is to handle new data, and it's possible for new data to contain categories that were not present in the training data. The argument `handle_unknown='ignore'` instructs the pipeline to ignore any categories that weren't present in the training data. For example, for new data we observe a different type of `style`. We don't want the code to break, so `handle_unknown='ignore'` will ignore these rows.

In a later section of the lab you'll need to use one-hot encoding to derive features from textual data. Importantly, you'll need to use one-hot encoding within a pipeline on Problem Set 3.

## Part 2 - Handling Missing Data

While not directly related to the topic of feature engineering. It's worthwhile knowing that pipelines provide an opportunity to handle missing data without introducing data leakage. This section of the lab introduces two different missing data imputation methods: simple imputation and nearest neighbors imputation, but there are many other strategies that exist.

The goal of imputation is to "fill in" the missing values in a data set so that all of its rows can be used during model training/evaluation. Many machine learning algorithms in `sklearn` cannot use data-points with missing values.

To demonstrate imputation, we'll load the "Happy Planet" data set, which contains various measures of different countries around the world. Let's try to predict the happiness score of a country using it's region, life expectancy, and human development index (HDI). We're primarily interested in the fact that two countries have missing values for the variables `GDPperCapita` and `HDI`.



In [ ]:
# Import data set w/ missing values
hp = pd.read_csv(filepath_or_buffer = "../data/HappyPlanet.csv")
hp.head()

In [ ]:
# Tally number of missing values (by variable)
hp.isna().sum()

In [ ]:
# Outcome
hp_y = hp['Happiness']

# Predictors
hp_X = hp[['Region','LifeExpectancy', 'HDI']]

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

simple_pipe = Pipeline(steps = [
    ("model", KNeighborsRegressor())
])

# The missing values cause an error here:
simple_fit = simple_pipe.fit(X = hp_X, y = hp_y)
simple_fit.score(X = hp_X, y = hp_y)

Below we create two different pipelines:

1. the first imputes missing values using the mean value of the variable containing the missing data. Theoretically speaking, simple mean imputation is only a valid approach when missing values are "missing completely at random", or when the missing values appear entirely at random in the data with no relation to the values of other observed variables or the identity of the involved rows.
2. the second imputes values using $k$-nearest neighbors, an approach that is valid when missing values are "missing at random", or when their occurrence can be explained using other variables observed in the data.


Tangent: there are different classes of missingness

- Under the "missing completely at random" (MCAR) definition, the United States and Afghanistan would be equally likely to have missing data (which probably isn't true)
- Under the "missing at random" (MAR) definition, missing information is predictable using other observed variables, so the United States and Afghanistan can differ in their propensity for missing data so long as the missing data can be explained by other observed variables.
- There is another possibility, "missing not at random" (MNAR), where missing information is only predictive using unobserved factors. Under these circumstances imputing missing data isn't justifiable from a statistical perspective, and might cause problems for the performance of machine learning models due to the imputed values being biased.

Determining whether your missing values are MCAR, MAR, or MNAR generally requires domain-specific knowledge about your data and how it was collected, thereby complicating the decision whether to include missing data imputation as part of a pre-processing pipeline.

In [ ]:
# Import imputation functions
from sklearn.impute import KNNImputer, SimpleImputer

simple_imp_pipe = Pipeline(steps = [
    ("imputer", SimpleImputer(missing_values=np.nan, strategy='mean')),
    ("model", KNeighborsRegressor())
])

simple_imp_fit = simple_imp_pipe.fit(X = hp_X, y = hp_y)
# The default scoring method for KNeighborsRegressor() is R^2
# https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html
simple_imp_fit.score(X = hp_X, y = hp_y)

Instead of filling every missing value with the overall mean, `KNNImputer` fills each missing value using the *k* most similar rows. For each row with a missing value, it finds its `n_neighbors=4` nearest rows, measured on the variables that aren't missing, and averages their values for the missing variable. With `weights='distance'`, closer neighbors count more.

Analogy: to guess a country's missing HDI:
1. mean imputation uses the world average
1. KNN imputation uses the HDI of the four countries that look most like it (similar region and life expectancy).

Like the mean imputer, the neighbors are found using the training data only inside the pipeline, so no information leaks from the validation fold.


In [ ]:
knn_imp_pipe = Pipeline(steps = [
    ("imputer", KNNImputer(missing_values=np.nan, n_neighbors=4, weights='distance')),
    ("model", KNeighborsRegressor())
])

kp_fit = knn_imp_pipe.fit(X = hp_X, y = hp_y)
kp_fit.score(X = hp_X, y = hp_y)

Because there were only two missing values in this example, there's seemingly not much difference in the training scores ($R^2$) between these two strategies.

However, something to recognize is that we treated the "Region" variable as numeric, when it actually uses integer values to encode different categories. Additionally, we did not perform any standardization or scaling in our pipeline, which will impact `KNNImputer()`.

**Question 1**: In this question you will improve upon the example pipeline from this section.

- **Part A**: Modify the example pipeline using KNN imputation to include the following changes:
    - Handle categorical and numerical variables separately within the pipeline:
        - The categorical variable `'Region'` should be transformed using one-hot encoding
        - The numeric predictors `'LifeExpectancy'` and `'HDI'` should be standardized
    - Impute any missing values using KNN
    - Use a Random Forest with `max_depth=2` and `n_estimators = 50` as the final model
- **Part B**: Fit your pipeline to the full training set and compare the method's score ($R^2$, the default metric) with the example. 

In [ ]:
# Code for Q1.a) 
SEED = 76

from sklearn.ensemble import RandomForestRegressor

num_transformer = Pipeline(steps = [
    ("scaler", StandardScaler()),
    ("imputer", KNNImputer(missing_values=np.nan, n_neighbors=4, weights='distance'))
])

cat_transformer = Pipeline(steps = [
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

num_cols = ['LifeExpectancy', 'HDI']
cat_cols = ['Region']

# Preprocessing transformer allowing different actions for numeric and categorical vars
preprocessor = ColumnTransformer(transformers = [
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

final_pipe = Pipeline(steps = [
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(max_depth = 2, n_estimators = 50, random_state = SEED))
])

# Fit new pipeline and print the score
kpn_fit = final_pipe.fit(X = hp_X, y = hp_y)
kpn_fit.score(X = hp_X, y = hp_y)

In [ ]:
# Code for Q1.b)
print(kp_fit.score(X = hp_X, y = hp_y))
print(kpn_fit.score(X = hp_X, y = hp_y))

## (Optional) Part 3 - Text Data

The data we'll use for the next portion of the lab come from the [UC-Irvine Machine Learning Repository](https://archive-beta.ics.uci.edu/datasets) and contain 5574 SMS text messages that are labeled as either "spam" or "ham" (not spam).

In [ ]:
# Note that textfile containing these data uses a tab delimiter to separate the label and message
sms = pd.read_csv(filepath_or_buffer = "../data/sms_spam.txt", sep='\t', names=['Label','Message'])
sms.head()

These data have a clear outcome, `Label`, but the predictive features must be derived from the message.

You might immediately want to jump to [tokenizing](https://nlp.stanford.edu/IR-book/html/htmledition/tokenization-1.html) each unique word in the message to be a predictive feature, but this brute force approach can often be beat using domain knowledge and feature engineering. This is because tokenization creates a high-dimensional feature space where many of these features are useless for prediction. So, unless you have a massive amount of data, overfitting is a major problem with such an approach.

To see this, let's start by creating a training set and looking at a few messages that are labeled "spam":

In [ ]:
# Train/test split
train_sms, test_sms = train_test_split(sms, test_size=0.2, random_state=8)

# Separate outcome from message and then convert it to 0/1 binary
train_sms_y = (train_sms['Label'] == 'spam').astype(dtype = int)
train_sms_X = train_sms['Message']

# Print some examples of 'spam'
print(train_sms_y.head())
print("\n")
print(train_sms_X[train_sms_y ==1].sample(n = 10))

Inspecting these spam messages provides a several ideas for feature engineering. For example, we can notice that spam messages tend to contain a lot of numbers and may also contain an inordinate number of exclamation marks.  

Based upon these observations we might engineer two features:
1. one measuring the proportion of characters in a message that are numeric
1. another counting the number of exclamation marks in the message.

In [ ]:
# Function to measure percentage of numbers
def get_num(text):
    return sum(map(str.isdigit, text))/len(text)

# Function to count exclamation marks
def count_exclamation_points(text):
    return text.count('!')

# Create a dictonary, convert to a Pandas dataframe, and display
d = {
    'prop_num': train_sms_X.apply(func = get_num), 
    'num_exclam': train_sms_X.apply(func = count_exclamation_points)
}
train_X = pd.DataFrame(data = d)
train_X.head(n = 6)

We can see how these features might be useful in classifying each message:

In [ ]:
# Scatter plot colored by label
plt.figure().set_figheight(val = 5)
plt.scatter(x = train_X['prop_num'], y = train_X['num_exclam'], c = train_sms_y)
plt.xlabel(xlabel = "Proportion numerical")
plt.ylabel(ylabel = "Number of exclamation points")
plt.show()

Are the purple or yellow dots spam? Investigating the single message where `prop_num` is 1, it seems purple are the non-spam messages:

In [ ]:
train_sms[train_X['prop_num'] == 1]

It's important to recognize that these are just two simple examples of features you might engineer from these messages. There are countless possibilities that could be even better predictors of "spam".

Additionally, it's beyond the scope of this course to thoroughly cover the topic of string processing Python, but I encourage you to [consult this reference](https://www.w3schools.com/python/python_ref_string.asp) for guidance and examples when trying to write your own functions to engineer features from text.

**Question 2**: In this question you'll continue working with the SMS spam data. You are encouraged to use this as a starting point for Problem Set 3.

- **Part A**: Import the "regular expression operations" library using `import re`, then use the `findall()` function with the regular expression `r'[A-Z]'` to create a function that counts the percentage of a message's characters that are capital letters.
- **Part B**: The function `first_word()` that is defined below will return the first word in a message. It works by splitting the string at the character `' '` (space), keeping only the first word (position 0), converting this word to lower case, and then removes any exclamation points by replacing them with `''` or an empty string. Use this function and the function you created in Part A to assemble a data frame containing two engineered features named "prop_cap" and "first_word".
- **Part C**: Determine two words that appear in a large share of "spam" messages **but do not appear in a similarly large share of "ham" messages**. *Hints*:
    - Use the `value_counts()` method on separate segments of the training data. You might also want to use the `head()` method to avoid printing a large table of counts.
    - `NaN` means the data is missing
- **Part D**: Apply one-hot encoding to the "first_word" column in the data frame from Part B, then keep only the columns that encode the presence of the two words you identified in Part C.
- **Part E**: Train a classification model of your choosing on the data from Part D. This data frame should include 3 predictive features: the percentage of capital letters (Part A) and the one-hot encoded first words (Parts B-C). Use a cross-validated grid search to find the optimal values of any hyperparameters. Report the best model and its cross-validated F1-score.
- **Part F**: Provide a short explanation for why the F1-score is a good way to evaluate model performance in this application.

In [ ]:
# Code for Q2.a)
import re
def get_prop_cap(text):
    return len(re.findall(pattern = r'[A-Z]', string = text))/len(text)

print(get_prop_cap(text = "ABCDdef"))

In [ ]:
# Code for Q2.b)
def first_word(text):
    return text.split(sep=' ')[0].lower().replace('!','')

print(first_word(text = "Florence is not the capital of Italy."))

In [ ]:
d = {
    'prop_cap': train_sms_X.apply(func = get_prop_cap),
    'first_word': train_sms_X.apply(func = first_word)
}
train_X = pd.DataFrame(data = d)
train_X.head()

In [ ]:
# Code for Q2.c)
word_frequencies = pd.concat(
    objs = [train_X[train_sms_y == 1]['first_word'].value_counts().head(n = 10), 
     train_X[train_sms_y == 0]['first_word'].value_counts().head(n = 10)], 
    axis = 1)
word_frequencies.columns = ['Spam', 'Ham']
word_frequencies

Observe that "urgent" and "free" are words that frequently occur in spam messages, but not ham ones. "you" is a word that occurs frequently in both, so it doesn't tell you anything about the likelihood that a message is spam

In [ ]:
# Code for Q2.d)
from pandas import get_dummies
train_X = get_dummies(data = train_X, columns = ['first_word'])[['prop_cap', 'first_word_urgent', 'first_word_free']]
train_X.head()

In [ ]:
# Code for Q2.e)
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

my_pipe = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])
my_params = {
    'model__n_neighbors': [5,10,15],
    'model__weights': ['distance', 'uniform']
}

grid_res = GridSearchCV(
    estimator = my_pipe, 
    param_grid = my_params, 
    cv=5, 
    scoring = "f1").fit(X = train_X, y = train_sms_y)

print(grid_res.best_estimator_)
print("\n")
print(grid_res.best_params_)
print("\n")
print(grid_res.best_score_)

**Answer for Q2.f)**

The F1 score is ideal here because accurately classifying spam, the positive class, is the focus of the application. The data have class imbalances and classifying 'spam' correctly is the goal

## (Optional) Part 4 - Time Series Classification

The data below were collected using a chest-mounted accelerometer in a research study on activity recognition. The original study collected data on 15 participants performing 7 activities, but for demonstration purposes we will only use data for 2 study participants and 2 activities: "working at a computer", or `'Work' `and "talking while standing", or `'Talk'`.

These data are uncalibrated accelaration measurements recorded at a rate of 52 Hz (so 52 measurements are recorded in each second). In our analysis we will look to classify 10-second samples of data, or collections of 520 inputs.

We will use the data from one participant (#1) for training, and we will evaluate the generalizability of our models using data from the other participant (#7) as a test set.

In [ ]:
# Dataset
act_ts = pd.read_csv(filepath_or_buffer = "../data/activity.csv")

# Training Data
train_X = act_ts[act_ts['Subject'] == 1].drop(labels = ['Label'], axis = 1)
train_y = act_ts[act_ts['Subject'] == 1]['Label']

# Validation Data
test_X = act_ts[act_ts['Subject'] == 7].drop(labels = ['Label'], axis = 1)
test_y = act_ts[act_ts['Subject'] == 7]['Label']

print(train_X.head())
print("\n")
print(train_y.head())

Successful feature engineering begins with a thoughtful examination of the training data in order to find predictive patterns. It is essential that this exploration only involves the training data to avoid data leakage.

Below we'll create a few simple graphs of the three available predictors over time, colored by the label:

In [ ]:
my_vars = ['X_accel', 'Y_accel', 'Z_accel']
for cur_var in my_vars:
    plt.figure().set_figheight(val = 2)
    plt.scatter(
        x = range(len(train_X[cur_var])), 
        y=train_X[cur_var], 
        c = train_y.map(arg = {'Work': 1, 'Talk': 0})
    )
    plt.title(label = cur_var)
    plt.show()

There are many feature engineering ideas we can glean from this exploration, below are two that stand out:

1. In general, working tends to show a lower amount of variability in the accelerometer's values across all axes.
2. The accelerometer tends to have higher values for working in the X and Z axes.

To translate these ideas into useable features we can use grouped summarization:

In [ ]:
train_X_FE = pd.concat(
    objs = [train_X.groupby(by = 'Sample_Num')[my_vars].mean(),
     train_X.groupby(by = 'Sample_Num')[my_vars].std()], 
    axis=1)

train_X_FE.columns = ['Mean_X', 'Mean_Y', 'Mean_Z', 'SD_X', 'SD_Y', 'SD_Z']
train_X_FE.head(n = 5)

We can confirm that these simple features are incredibly predictive, at least on the training data: 

In [ ]:
train_y_FE = act_ts[act_ts['Subject'] == 1].groupby(by = 'Sample_Num')['Label'].first()

plt.scatter(
    x = train_X_FE['Mean_X'], 
    y = train_X_FE['SD_X'], 
    c = train_y_FE.map(arg = {'Work': 1, 'Talk': 0})
)
plt.show()

**Question 3**: For this question you should continue using the data introduced in this section's example.

- **Part A**: Train a support vector machine classifier using the 6 engineered features described in the example given above. Choose an appropriate kernel based upon the relationships you can visually see in the data that is also justified by cross-validation. Report the cross-validated classification accuracy of this approach.
- **Part B**: Add a standardization step to the approach you explored in Part A and adjust any tuning parameters as necessary. Does this change make any difference in the classification accuracy achieved *on the test data*? Briefly explain what you observe.

In [ ]:
# Code for Q3.a)
from sklearn.svm import SVC

my_pipe = Pipeline(steps = [
    ('model', SVC())
])

my_params = {
    'model__kernel': ['poly','linear'],
    'model__C': [0.001,0.1]
}

grid_res = GridSearchCV(
    estimator = my_pipe, 
    param_grid = my_params, 
    cv=5).fit(X = train_X_FE, y = train_y_FE)

print(grid_res.best_estimator_)
print("\n")
print(grid_res.best_params_)
print("\n")
print(grid_res.best_score_)

In [ ]:
# Code for Q3.b)
test_X_FE = pd.concat(
    objs = [test_X.groupby(by = 'Sample_Num')[my_vars].mean(),
     test_X.groupby(by = 'Sample_Num')[my_vars].std()],
    axis=1)
test_y_FE = act_ts[act_ts['Subject'] == 7].groupby(by = 'Full_Sample_ID').first()['Label']

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler

my_pipe2 = Pipeline(steps = [
    ('scaler', MinMaxScaler()),
    ('model', SVC())
])

my_params = {
    'model__kernel': ['poly','linear'],
    'model__C': [0.01,0.1]
}

grid_res2 = GridSearchCV(
    estimator = my_pipe2, 
    param_grid = my_params, 
    cv=5).fit(X = train_X_FE, y = train_y_FE)

print(grid_res2.best_estimator_)
print("\n")
print(grid_res2.best_params_)
print("\n")
print(grid_res2.best_score_)

In [ ]:
test_pred_y = grid_res.predict(X = test_X_FE)
test_pred_y2 = grid_res2.predict(X = test_X_FE)
print(accuracy_score(y_true = test_y_FE, y_pred = test_pred_y), accuracy_score(y_true = test_y_FE, y_pred = test_pred_y2))

**Question 4**: The data below are a different portion activity recognition study described at the beginning of this section for *3 subjects* and two different activities: "Walking", or `'Walking'`, and "Walking and Talking to Someone", or `'Walking_Talking'`.

- **Part A**: Split the data to use the subjects #5 and #7 as training data and subject #1 as test data.
- **Part B**: Create at least one data visualization and use it to engineer at least two features derived from the variables `X_accel`, `Y_accel`, and `Z_accel`. Your answer to this question should include your visualization and code that creates a version of your data with 1 row per sample with your engineered features as columns (similar to the format of the example given in this part of the lab).
- **Part C**: Create a pipeline involving at least one preprocessing step (ie: standardization/re-scaling, normalizing transformation, dimension reduction, etc.) followed by a classifier step. Use a cross-validated grid search to tune at least one hyperparameter involved in this pipeline.
- **Part D**: Evaluate your best performing method on the test set and briefly comment upon how successful you were.

~~*Hint*: Classifying these activities is significantly more difficult than the example distinguishing between working and talking. The purpose of this exercise is for you to explore the creation of more sophisticated features (beyond simple means/standard deviations). Some possible ideas for features are rolling averages, ranges, rates of change, etc. You should not be discouraged if you aren't able to achieve a very high degree of classification accuracy.~~

In [ ]:
# Question 4 Dataset
act_ts_v2 = pd.read_csv(filepath_or_buffer = "../data/activity_v2.csv")

In [ ]:
# Code for Q4.a)
q3_X_ts_train = act_ts_v2[act_ts_v2['Subject'] != 1]
q3_X_ts_test = act_ts_v2[act_ts_v2['Subject'] == 1]

In [ ]:
# Code for Q4.b)
my_vars = ['Full_Sample_ID','X_accel', 'Y_accel', 'Z_accel']
max1s_ch = q3_X_ts_train[my_vars].groupby(by = 'Full_Sample_ID').rolling(window = 26, min_periods = 1).mean().groupby(by = 'Full_Sample_ID').max()
min1s_ch = q3_X_ts_train[my_vars].groupby(by = 'Full_Sample_ID').rolling(window = 26, min_periods = 1).mean().groupby(by = 'Full_Sample_ID').min()
train_X = pd.concat(
    objs = [max1s_ch.rename(columns={'X_accel': 'X_max', 'Y_accel': 'Y_max', 'Z_accel': 'Z_max'}),
     min1s_ch.rename(columns={'X_accel': 'X_min', 'Y_accel': 'Y_min', 'Z_accel': 'Z_min'})], 
    axis = 1)
train_y = act_ts_v2[act_ts_v2['Subject'] != 1].groupby(by = 'Full_Sample_ID').first()['Label']

plt.scatter(x = train_X['Y_max'], y = train_X['Y_min'], c=train_y.map(arg = {'Walking': 0, 'Walking_Talking': 1}))

In [ ]:
# Code for Q4.c)
# Example pipeline
my_pipe = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('model', SVC(kernel = 'rbf', probability=True))
])

my_params = {'model__C': [1,10]}

grid_res = GridSearchCV(
    estimator = my_pipe, 
    param_grid = my_params, 
    cv=5).fit(X = train_X, y = train_y)

print(grid_res2.best_estimator_)
print("\n")
print(grid_res2.best_params_)
print("\n")
print(grid_res2.best_score_)

In [ ]:
# Code for Q4.d)
max1s_ch2 = q3_X_ts_test[my_vars].groupby(by = 'Full_Sample_ID').rolling(window = 26, min_periods = 1).mean().groupby(by = 'Full_Sample_ID').max()
min1s_ch2 = q3_X_ts_test[my_vars].groupby(by = 'Full_Sample_ID').rolling(window = 26, min_periods = 1).mean().groupby(by = 'Full_Sample_ID').min()

test_X = pd.concat(
    objs = [max1s_ch2.rename(columns={'X_accel': 'X_max', 'Y_accel': 'Y_max', 'Z_accel': 'Z_max'}),
     max1s_ch2.rename(columns={'X_accel': 'X_min', 'Y_accel': 'Y_min', 'Z_accel': 'Z_min'})], 
    axis = 1)
test_Y = act_ts_v2[act_ts_v2['Subject'] == 1].groupby(by = 'Full_Sample_ID').first()['Label']

test_pred_q3 = grid_res.predict(X = test_X)
print(accuracy_score(y_true = test_Y, y_pred = test_pred_q3))